# White advantage on Lichess: run in Colab

Streams the Lichess monthly files straight from database.lichess.org into `white_advantage`,
so nothing big is saved to disk and memory use stays small.

This is the **one slow run**. It saves `counts.csv`: every game counted by opening, speed,
rating bracket and rating gap. Everything else (tables, tests, the dashboard and the final
report) is rebuilt from that file in seconds with `--from-counts`, so you never need to
read the 190 million games again.

Run the cells in order. At the end you download `results.zip`: put everything in it into
`output/final/` in the repo and push. (`output/checkpoint1/` keeps the Checkpoint 1 results.) Keep this tab open while it runs.

## 1. Install zstd and compile the program

In [ ]:
!apt-get -qq update > /dev/null
!apt-get -qq install -y libzstd-dev > /dev/null
!rm -rf repo && git clone -q https://github.com/Sahil-M-Deo/The-Perks-of-Being-a-First-Mover.git repo
!g++ -O2 -std=c++17 -pthread repo/scripts/white_advantage.cpp -o repo/scripts/white_advantage -lzstd && echo "compiled"

## 2. Choose the months

`MAX_GAMES = 0` reads every game. Run once with `MAX_GAMES = 1000000` first: it takes a few
minutes and checks that everything works (see the check under step 3).

In [ ]:
MONTHS = ["2024-01", "2024-02"]
MAX_GAMES = 0

## 3. Run

Each month is downloaded and read at the same time through a pipe (`mkfifo`),
so the compressed file never sits on disk. The full run is limited by download speed.

**Check after it finishes:**
- the "games read" for each month should match the Games column on database.lichess.org
  (for a test run, it should equal `MAX_GAMES`). If it's lower, the download was cut off; rerun.
- in the "By opening" table, every opening should have games. A line saying
  "no games (check the name...)" means Lichess spells that opening differently.
- on a small test run some of the tests print `nan`: there aren't enough games per rating
  bracket yet. That's expected and goes away in the full run.

In [ ]:
files = [f"lichess_db_standard_rated_{m}.pgn.zst" for m in MONTHS]
lines = ["mkdir -p repo/output/final", "cd repo/output/final", "rm -f lichess_db_*.pgn.zst"]
for f in files:
    # a named pipe with the real file name, so the program knows the month and that it is .zst
    lines.append(f"mkfifo {f}")
    # -f: on an HTTP error write nothing, instead of feeding an HTML error page to zstd
    lines.append(f"curl -fsSL https://database.lichess.org/standard/{f} > {f} &")
extra = f"--max-games {MAX_GAMES}" if MAX_GAMES > 0 else ""
lines.append(f"../../scripts/white_advantage {' '.join(files)} {extra} | tee summary.txt")
lines.append("rm -f lichess_db_*.pgn.zst")
open("run.sh", "w").write("\n".join(lines) + "\n")

# run with ! (not subprocess) so the output shows up live in the notebook
!bash run.sh

## 4. Download the results

In [ ]:
!cd repo/output/final && zip -q ../../../results.zip counts.csv output.csv openings.csv summary.txt dashboard.html; ls -la ../../../results.zip
from google.colab import files as colab_files
colab_files.download("results.zip")

## 5. (Later) Rebuild the report and dashboard from `counts.csv`

Once `counts.csv` is pushed to `output/final/` in the repo, this rebuilds every table, test, the
dashboard and `finalReport.pdf` in seconds, without reading the games again. Run step 1 first
(it clones the latest repo), then this. Installing LaTeX takes a few minutes.

In [ ]:
!apt-get -qq install -y texlive-latex-recommended texlive-latex-extra texlive-fonts-recommended texlive-pictures > /dev/null && echo "LaTeX installed"
!cd repo/output/final && ../../scripts/white_advantage --from-counts counts.csv --report ../../reports/final/report_numbers.tex
!cd repo && zip -q -j ../report.zip reports/final/finalReport.pdf output/final/dashboard.html output/final/output.csv output/final/openings.csv
from google.colab import files as colab_files
colab_files.download("report.zip")